# Chapter 2 — Few-shot prompting (épée tactical recommender)

My own experiments applying the few-shot ideas from Chapter 2 of *AI Agents and Applications*.

**Task**: given an épée bout situation, recommend a tactical action in a strict format:

```
Action: ... | Distance: ... | Rationale: ...
```

**Why this task**: épée has a specific vocabulary (prise de fer, second intention, point in line, fleche, arrest, etc.) that base models *know* but tend to misuse. The output format is also strict. Both of those are exactly what few-shot prompting is good for — calibrating vocabulary and locking in format.

**What I'm going to do** (and watch for):
1. Zero-shot baseline — does the model already understand the task?
2. One-shot — does a single example change the output?
3. Three-shot inline — pattern locks in, but the prompt is now a mess to maintain.
4. Refactor to `FewShotPromptTemplate` — same examples, but now *data*.
5. Add a fourth example without touching the prompt definition — the whole point of the refactor.

---

## Setup

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
# Low temperature keeps the format stable across runs so we can compare
# zero-shot vs one-shot vs three-shot fairly. Bump it later to see how
# robust the few-shot pattern is.

## The test situation

We'll feed the *same* situation to every variant (zero-shot, 1-shot, 3-shot, `FewShotPromptTemplate`) so the only thing changing is the prompt structure. This is how you isolate the variable you're studying.

Feel free to swap this for a situation from a recent bout of yours — the more grounded it is in real fencing, the easier it'll be to judge whether the model's recommendation is actually good.

In [2]:
TEST_SITUATION = (
    "Tied 5-5, 60s left in a DE bout to 15. "
    "Opponent has a long reach and has stop-hit my last three advances to the hand. "
    "They prefer to fence at long distance and rarely commit to a full attack."
)

## Experiment 0 — Zero-shot baseline

Start with no examples. Just describe the task in instructions and see what the model does.

**What to look for in the output:**
- Does it use real épée vocabulary, or generic "go for the touch" advice?
- Does it follow the format we asked for?
- Is the rationale grounded in fencing logic, or hand-wavy?

Don't skip this step in your own work — if zero-shot is already great, you're wasting tokens by adding examples.

In [3]:
zero_shot = PromptTemplate.from_template(
    "You are an épée fencing tactician. Given a bout situation, "
    "recommend an action in this exact format:\n"
    "Action: ... | Distance: ... | Rationale: ...\n\n"
    "Situation: {situation}\n→"
)

chain = zero_shot | llm
print("--- zero-shot ---")
print(chain.invoke({"situation": TEST_SITUATION}).content)

--- zero-shot ---
Action: Initiate a feint to the high line followed by a disengage to the low line | Distance: Close to mid-distance | Rationale: By feinting to the high line, you can provoke your opponent's stop-hit response, which they have been relying on. The disengage to the low line will allow you to bypass their defense and target their hand or lower opening, taking advantage of their tendency to maintain distance and not fully commit to attacks. This approach can create an opportunity to score while minimizing the risk of being hit.


**Pause and observe.** Write down (mentally or in a markdown cell below) what's off about that output. Common zero-shot failures here:
- Vocabulary that sounds plausible but isn't quite right
- Format drift (missing pipes, extra prose, no `→` marker)
- Generic tactics ("be patient and look for an opening") instead of named actions

## Experiment 1 — One-shot (a single example)

Now we hand-craft *one* example and bake it into the prompt. Watch how just one well-chosen demonstration changes the output.

In [4]:
one_shot = PromptTemplate.from_template(
    "You are an épée fencing tactician. Given a bout situation, "
    "recommend an action in this exact format:\n"
    "Action: ... | Distance: ... | Rationale: ...\n\n"
    "Situation: Down 2-3, 1:30 left. Opponent keeps lunging deep to my chest "
    "from middle distance. I'm slow to retreat.\n"
    "→ Action: arrest to forearm | Distance: hold middle | Rationale: their "
    "committed attack passes through my point line; the forearm scores before "
    "their lunge lands.\n\n"
    "Situation: {situation}\n→"
)

chain = one_shot | llm
print("--- one-shot ---")
print(chain.invoke({"situation": TEST_SITUATION}).content)

--- one-shot ---
Action: disengage to counter-attack | Distance: close the distance slightly | Rationale: by disengaging, I can bypass their extended point and create an opening for a counter-attack to their upper target, exploiting their tendency to stay at long distance and not fully commit to an attack. This will also put pressure on them to react, potentially forcing a mistake.


**Pause and observe.** Compared to zero-shot:
- Is the format tighter?
- Is the vocabulary more specific?
- Did the model pick up the *style* of the rationale (concrete cause → effect)?

Often a single example is enough to lock in the format, but the *content* is still inconsistent. That's where adding more examples helps — not to teach the format harder, but to teach the model the *space* of acceptable answers.

## Experiment 2 — Three-shot inline (and why it hurts)

Now we add two more examples that vary across the input space (different score states, opponent patterns, time pressure). Watch the output get sharper — *and* watch the prompt string get unwieldy.

In [5]:
three_shot = PromptTemplate.from_template(
    "You are an épée fencing tactician. Given a bout situation, "
    "recommend an action in this exact format:\n"
    "Action: ... | Distance: ... | Rationale: ...\n\n"
    "Situation: Down 2-3, 1:30 left. Opponent keeps lunging deep to my chest "
    "from middle distance. I'm slow to retreat.\n"
    "→ Action: arrest to forearm | Distance: hold middle | Rationale: their "
    "committed attack passes through my point line; the forearm scores before "
    "their lunge lands.\n\n"
    "Situation: Tied 4-4, 30s left. Opponent has been parrying my high-line "
    "attacks all bout. They're cautious about double-touches.\n"
    "→ Action: feint high, disengage to wrist | Distance: long lunge | "
    "Rationale: their parry reflex opens the wrist on the disengage, and the "
    "wrist target avoids the double-touch risk a chest attack carries.\n\n"
    "Situation: Leading 5-3, 45s left. Opponent is aggressive and keeps "
    "fleching at me. I want to play out the clock.\n"
    "→ Action: point in line, retreat | Distance: maximum | Rationale: line "
    "forces them to deal with my point before launching; retreating burns "
    "clock without conceding the bout.\n\n"
    "Situation: {situation}\n→"
)

chain = three_shot | llm
print("--- three-shot ---")
print(chain.invoke({"situation": TEST_SITUATION}).content)

--- three-shot ---
Action: advance with a low line attack to the foot | Distance: close | Rationale: by targeting the foot, I can bypass their reach advantage and disrupt their rhythm, forcing them to react to my attack rather than initiating their own. This approach minimizes the risk of their stop-hit while also creating an opportunity to score.


**Pause and observe two things now:**

1. **The output**: it should be visibly closer to the style and vocabulary of the examples. The format should be rock-solid.
2. **The prompt string**: it's already painful. Imagine you want to swap one example, or add a fourth, or load examples from a file. You're editing one giant string. This is exactly the same pain that motivated `PromptTemplate` in Chapter 1 — and now we feel it again at a higher level.

Time to refactor.

## Experiment 3 — `FewShotPromptTemplate` (the LangChain way)

The refactor pulls three things apart:

1. **`examples`** — a list of dicts. Pure data. Could come from a file, a DB, or be generated.
2. **`example_prompt`** — a tiny `PromptTemplate` that says how to *format one example*.
3. **The wrapper** (`prefix` + `suffix`) — the instructions before all examples and the user input after them.

`FewShotPromptTemplate` glues them together: it formats every example with `example_prompt`, joins them with newlines, sandwiches them between `prefix` and `suffix`, and returns one final string ready for the LLM.

In [ ]:
# 1. Examples as data
examples = [
    {
        "situation": "Down 2-3, 1:30 left. Opponent keeps lunging deep to my chest from middle distance. I'm slow to retreat.",
        "recommendation": "Action: arrest to forearm | Distance: hold middle | Rationale: their committed attack passes through my point line; the forearm scores before their lunge lands.",
    },
    {
        "situation": "Tied 4-4, 30s left. Opponent has been parrying my high-line attacks all bout. They're cautious about double-touches.",
        "recommendation": "Action: feint high, disengage to wrist | Distance: long lunge | Rationale: their parry reflex opens the wrist on the disengage, and the wrist target avoids the double-touch risk a chest attack carries.",
    },
    {
        "situation": "Leading 5-3, 45s left. Opponent is aggressive and keeps fleching at me. I want to play out the clock.",
        "recommendation": "Action: point in line, retreat | Distance: maximum | Rationale: line forces them to deal with my point before launching; retreating burns clock without conceding the bout.",
    },
]

# 2. How to render *one* example
example_prompt = PromptTemplate.from_template(
    "Situation: {situation}\n→ {recommendation}"
)

# 3. Glue: prefix (instructions) + examples + suffix (user's actual question)
few_shot = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=(
        "You are an épée fencing tactician. Given a bout situation, "
        "recommend an action in this exact format:\n"
        "Action: ... | Distance: ... | Rationale: ..."
    ),
    suffix="Situation: {situation}→\n",
    input_variables=["situation"],
    example_separator="\n\n",
)

Before we run it, let's peek at the *rendered* prompt. This is the whole point of `FewShotPromptTemplate` — it produces the same string we were hand-crafting in Experiment 2, but assembled from parts.

In [23]:
print(few_shot.format(situation=TEST_SITUATION))

You are an épée fencing tactician. Given a bout situation, recommend an action in this exact format:
Action: ... | Distance: ... | Rationale: ...

Situation: Down 2-3, 1:30 left. Opponent keeps lunging deep to my chest from middle distance. I'm slow to retreat.
 Action: arrest to forearm | Distance: hold middle | Rationale: their committed attack passes through my point line; the forearm scores before their lunge lands.

Situation: Tied 4-4, 30s left. Opponent has been parrying my high-line attacks all bout. They're cautious about double-touches.
 I'd feint high then disengage to the wrist. Long lunge. The parry reflex opens the wrist on the disengage.

Situation: Leading 5-3, 45s left. Opponent is aggressive and keeps fleching at me. I want to play out the clock.
 Action: point in line, retreat | Distance: maximum | Rationale: line forces them to deal with my point before launching; retreating burns clock without conceding the bout.

Situation: Tied 5-5, 60s left in a DE bout to 15. O

Now run it through the model. Output should match (or beat) the three-shot inline version from Experiment 2.

In [24]:
chain = few_shot | llm
print("--- FewShotPromptTemplate ---")
print(chain.invoke({"situation": TEST_SITUATION}).content)

--- FewShotPromptTemplate ---
Action: advance with a short lunge to the body | Distance: close | Rationale: closing the distance disrupts their preferred long-range strategy and minimizes their ability to stop-hit; the short lunge targets their body directly, increasing the chance of scoring while forcing them to react quickly.


## Experiment 4 — Add an example without touching the prompt

This is the payoff. Suppose you want to add a fourth example covering a *new* tactical pattern — say, second-intention attacks. With the inline three-shot version, you'd be editing a giant string. Here, you just append to the `examples` list.

In [ ]:
examples.append({
    "situation": "Tied 6-6, plenty of time. Opponent reliably parries-ripostes any committed attack to their chest.",
    "recommendation": "Action: second-intention — false attack, draw the riposte, parry-counter | Distance: lunge depth, prepare to recover | Rationale: their riposte is predictable; the false attack lets me set up the parry I actually want to score on.",
})

# Re-build with the updated examples list — no prompt string changes
few_shot_v2 = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=few_shot.prefix,
    suffix=few_shot.suffix,
    input_variables=["situation"],
    example_separator="\n\n",
)

chain = few_shot_v2 | llm
print("--- FewShotPromptTemplate, 4 examples ---")
print(chain.invoke({"situation": TEST_SITUATION}).content)

## Takeaways

- **Zero-shot first, always.** If it works, you're done. Few-shot costs tokens and complexity.
- **One good example > three mediocre ones.** Pick examples that span the input space, not three near-duplicates.
- **Inline few-shot is fine for a quick experiment. `FewShotPromptTemplate` is what you want once you care about iteration.** Examples become *data*: addable, removable, loadable from JSON, swappable per request.
- **The pattern is the same as Chapter 1.** Pull strings out of code, give them structure, gain the ability to vary the data without touching the template.
- **Format vs content.** Few-shot is great at locking in *output format* and *vocabulary*. It's weaker at teaching brand-new reasoning the model couldn't already do — examples calibrate, they don't fundamentally extend capability.

## What's next to try

- **`FewShotChatMessagePromptTemplate`** — the chat-native version, where each example becomes a (user, assistant) message pair instead of a flat string. Better for chat models like GPT-4o-mini.
- **Example selectors** — instead of always sending all four examples, dynamically pick the most relevant 2 for each new input using `SemanticSimilarityExampleSelector`. Useful when your example library grows past ~10.
- **Load examples from a file** — drop the list into `examples.json`, parse it at import time. Now you can edit examples without touching code.